# Concept Erasure

**Survey Reference:** Section 13.1 - Concept Erasure: Removing Unwanted Concepts

## Overview

**Concept erasure** removes specific concepts from model representations, making them inaccessible even to probes. Key methods:

- **LEACE (Least-squares Concept Erasure):** Optimal linear erasure
- **PISCES:** Iterative shrinkage-based erasure
- **Fine-tuning approaches:** Train to forget specific information

### Applications

- Remove protected attributes (gender, race) for fairness
- Erase harmful knowledge (e.g., bioweapon synthesis)
- Remove personal information for privacy

## Learning Objectives

1. Understand the concept erasure problem
2. Implement LEACE for linear erasure
3. Evaluate erasure effectiveness
4. Understand limitations and potential circumvention

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. The Concept Erasure Problem

Given:
- Representations $X \in \mathbb{R}^{n \times d}$
- Concept labels $Z \in \{0, 1\}^n$

Find transformation $P$ such that:
- $P(X)$ preserves useful information
- No linear classifier can recover $Z$ from $P(X)$

In [ ]:
# Create synthetic data with a concept we want to erase
np.random.seed(42)

n_samples = 1000
d = 50

# Concept direction (what we want to erase)
concept_direction = np.random.randn(d)
concept_direction = concept_direction / np.linalg.norm(concept_direction)

# Generate data with concept embedded
Z = np.random.randint(0, 2, n_samples)  # Binary concept labels
X_base = np.random.randn(n_samples, d)  # Base representations

# Add concept to representations
concept_strength = 2.0
X = X_base + concept_strength * np.outer(2*Z - 1, concept_direction)

print(f"Data shape: {X.shape}")
print(f"Concept distribution: {np.bincount(Z)}")

In [ ]:
def evaluate_concept_leakage(X, Z):
    """
    Measure how well the concept can be recovered from representations.
    """
    X_train, X_test, Z_train, Z_test = train_test_split(X, Z, test_size=0.2, random_state=42)
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, Z_train)
    
    accuracy = clf.score(X_test, Z_test)
    return accuracy

# Before erasure
original_leakage = evaluate_concept_leakage(X, Z)
print(f"Concept leakage before erasure: {original_leakage:.4f}")

## 2. LEACE: Least-squares Concept Erasure

LEACE finds the optimal projection that removes concept information while preserving the rest:

$$P = I - W(W^TW)^{-1}W^T$$

where $W$ is the direction that encodes the concept.

In [ ]:
class LEACE:
    """
    Least-squares Concept Erasure.
    
    Projects out the subspace that linearly predicts the concept.
    """
    
    def __init__(self):
        self.projection = None
    
    def fit(self, X: np.ndarray, Z: np.ndarray):
        """
        Fit the erasure projection.
        
        Args:
            X: [n_samples, d] representations
            Z: [n_samples] concept labels (binary)
        """
        # Center the data
        X_centered = X - X.mean(axis=0, keepdims=True)
        
        # Find direction that predicts concept (difference in means)
        pos_mean = X[Z == 1].mean(axis=0)
        neg_mean = X[Z == 0].mean(axis=0)
        concept_dir = pos_mean - neg_mean
        concept_dir = concept_dir / (np.linalg.norm(concept_dir) + 1e-8)
        
        # Create projection that removes this direction
        # P = I - v v^T (for unit vector v)
        self.concept_direction = concept_dir
        self.projection = np.eye(X.shape[1]) - np.outer(concept_dir, concept_dir)
        
        return self
    
    def transform(self, X: np.ndarray) -> np.ndarray:
        """Apply erasure projection."""
        return X @ self.projection.T
    
    def fit_transform(self, X: np.ndarray, Z: np.ndarray) -> np.ndarray:
        return self.fit(X, Z).transform(X)

In [ ]:
# Apply LEACE
leace = LEACE()
X_erased = leace.fit_transform(X, Z)

# Evaluate concept leakage after erasure
erased_leakage = evaluate_concept_leakage(X_erased, Z)
print(f"Concept leakage after LEACE: {erased_leakage:.4f}")
print(f"Reduction: {original_leakage - erased_leakage:.4f}")

## 3. Multi-dimensional Concept Erasure

What if the concept is encoded in multiple directions?

In [ ]:
class IterativeLEACE:
    """
    Iteratively erase concept until no linear probe can recover it.
    """
    
    def __init__(self, n_iterations: int = 5, tolerance: float = 0.55):
        self.n_iterations = n_iterations
        self.tolerance = tolerance
        self.projections = []
    
    def fit(self, X: np.ndarray, Z: np.ndarray):
        """Iteratively fit erasure projections."""
        X_current = X.copy()
        
        for i in range(self.n_iterations):
            # Check if concept is already erased
            leakage = evaluate_concept_leakage(X_current, Z)
            print(f"Iteration {i}: leakage = {leakage:.4f}")
            
            if leakage < self.tolerance:
                print(f"Concept sufficiently erased after {i} iterations")
                break
            
            # Fit and apply LEACE
            leace = LEACE()
            X_current = leace.fit_transform(X_current, Z)
            self.projections.append(leace.projection)
        
        return self
    
    def transform(self, X: np.ndarray) -> np.ndarray:
        """Apply all erasure projections."""
        X_current = X.copy()
        for proj in self.projections:
            X_current = X_current @ proj.T
        return X_current

# Test iterative erasure
iterative_leace = IterativeLEACE(n_iterations=5)
X_erased_iter = iterative_leace.fit(X, Z).transform(X)

final_leakage = evaluate_concept_leakage(X_erased_iter, Z)
print(f"\nFinal concept leakage: {final_leakage:.4f}")

## 4. Applying to Real Models

In [ ]:
# Load model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

def get_representation(model, tokenizer, text, layer_idx):
    """Get hidden state representation."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # Mean pool
    return outputs.hidden_states[layer_idx].mean(dim=1).cpu().numpy()

In [ ]:
# Example: Erase gender information
male_sentences = [
    "The man walked to the store.",
    "He is a doctor.",
    "The boy played in the park.",
    "His father works at the hospital.",
]

female_sentences = [
    "The woman walked to the store.",
    "She is a doctor.",
    "The girl played in the park.",
    "Her mother works at the hospital.",
]

# Collect representations
layer_idx = 6  # Mid layer

male_reps = np.vstack([get_representation(model, tokenizer, s, layer_idx) for s in male_sentences])
female_reps = np.vstack([get_representation(model, tokenizer, s, layer_idx) for s in female_sentences])

X_real = np.vstack([male_reps, female_reps])
Z_real = np.array([0] * len(male_sentences) + [1] * len(female_sentences))

print(f"Representations shape: {X_real.shape}")

In [ ]:
# Apply erasure
leace_real = LEACE()
X_real_erased = leace_real.fit_transform(X_real, Z_real)

# Note: With only 8 samples, accuracy evaluation is unreliable
# In practice, you'd use many more examples
print("Gender direction found with norm:", np.linalg.norm(leace_real.concept_direction))

## 5. Integrating Erasure into Model

In [ ]:
class ErasureHook:
    """
    Hook to apply concept erasure at inference time.
    """
    
    def __init__(self, projection: np.ndarray):
        self.projection = torch.tensor(projection, dtype=torch.float32).to(device)
    
    def __call__(self, module, input, output):
        hidden_states = output[0]
        # Apply projection to all positions
        # hidden_states: [batch, seq, hidden]
        erased = hidden_states @ self.projection.T
        return (erased,) + output[1:]


def generate_with_erasure(
    model,
    tokenizer,
    prompt: str,
    projection: np.ndarray,
    layer_idx: int,
    max_new_tokens: int = 50,
):
    """Generate with concept erasure applied."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    hook = ErasureHook(projection)
    handle = model.transformer.h[layer_idx].register_forward_hook(hook)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test generation with erasure
# prompt = "The doctor said that"
# result = generate_with_erasure(model, tokenizer, prompt, leace_real.projection, layer_idx)
# print(result)

## 6. Limitations and Circumvention

Important caveats:

1. **Nonlinear encoding:** Linear erasure may not catch nonlinear representations
2. **Multi-layer encoding:** Concept may be re-encoded in later layers
3. **Indirect inference:** Related concepts may still leak information
4. **Adversarial probes:** Stronger probes might recover erased information

In [ ]:
# TODO: Demonstrate limitations
# - Train nonlinear probe on erased representations
# - Check if concept reappears in later layers

## Key Takeaways

1. **LEACE** removes the linear subspace encoding a concept
2. **Iterative erasure** handles multi-dimensional concept encoding
3. **Integration** can be done via hooks at inference time
4. **Limitations** include nonlinear encoding and multi-layer redundancy
5. **Evaluation** should include strong probe attacks

## Next Steps

→ **02_steering_vectors.ipynb**: Control model behavior through activation interventions